In [ ]:
import torch
import torch.nn as nn
from transformers import CLIPProcessor, CLIPModel, AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
import json
from tqdm import tqdm
import random

In [ ]:
# --- 1. 설정 및 경로 정의 ---
print("설정을 시작합니다...")
# 모델 이름 (대회 규칙: 2024년 이전 공개, 오픈소스, 총 파라미터 3B 미만)
CLIP_MODEL_NAME = "openai/clip-vit-large-patch14" # 약 0.3B 파라미터
GPT2_MODEL_NAME = "gpt2-xl"                       # 약 1.558B 파라미터
# 총 파라미터: 0.3B + 1.558B = 1.858B. 매핑 네트워크 파라미터가 1.142B 미만이면 OK.

# VMCBench 데이터를 변환하여 저장한 절대 경로
VMCBENCH_BASE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/VMCBench_as_Official_VQAv2_9_1"

# 훈련 데이터 경로
TRAIN_ANNOTATION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_mscoco_train2014_annotations.json")
TRAIN_QUESTION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_OpenEnded_mscoco_train2014_questions.json")
TRAIN_IMAGE_DIR = os.path.join(VMCBENCH_BASE_DIR, "train2014")

# 검증 데이터 경로 (선택 사항이지만, 학습 모니터링에 강력 권장)
VAL_ANNOTATION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_mscoco_val2014_annotations.json")
VAL_QUESTION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_OpenEnded_mscoco_val2014_questions.json")
VAL_IMAGE_DIR = os.path.join(VMCBENCH_BASE_DIR, "val2014")


# 하이퍼파라미터
BATCH_SIZE = 2 # GPU 메모리 상황에 따라 조절. GPT-2 XL은 매우 큽니다.
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
NUM_SOFT_TOKENS = 20 # CLIP 임베딩이 변환될 가상 토큰의 개수. 10~50 사이에서 실험.

# 장치 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용할 장치: {device}")

In [ ]:
# --- 2. 모델 및 토크나이저 로드 ---
print("모델 및 토크나이저를 로드 중입니다...")
# CLIP 모델 로드 및 가중치 고정 (학습하지 않음)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
clip_model.eval() # 추론 모드
for param in clip_model.parameters():
    param.requires_grad = False
print(f"CLIP 모델 로드 완료. 파라미터 수: {sum(p.numel() for p in clip_model.parameters()) / 1e9:.3f}B")

# GPT-2 모델 로드 및 가중치 고정 (학습하지 않음)
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL_NAME).to(device)
gpt2_model.eval() # 추론 모드
for param in gpt2_model.parameters():
    param.requires_grad = False
# GPT-2 토크나이저에 패딩 토큰 설정 (일관성을 위해 EOS 토큰 사용)
if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model.config.pad_token_id = gpt2_tokenizer.pad_token_id
print(f"GPT-2 모델 로드 완료. 파라미터 수: {sum(p.numel() for p in gpt2_model.parameters()) / 1e9:.3f}B")

In [ ]:
# --- 3. 매핑 네트워크 정의 ---
# CLIP 임베딩을 GPT-2의 소프트 프롬프트 임베딩으로 변환하는 네트워크
class ImageToSoftPromptMapping(nn.Module):
    def __init__(self, clip_embedding_dim, gpt2_hidden_dim, num_soft_tokens):
        super().__init__()
        self.num_soft_tokens = num_soft_tokens
        # gpt2_hidden_dim을 인스턴스 변수로 저장
        self.gpt2_hidden_dim = gpt2_hidden_dim # 이 줄을 추가합니다.
        self.mlp = nn.Sequential(
            nn.Linear(clip_embedding_dim, gpt2_hidden_dim * num_soft_tokens // 2),
            nn.ReLU(),
            nn.Linear(gpt2_hidden_dim * num_soft_tokens // 2, gpt2_hidden_dim * num_soft_tokens)
        )

    def forward(self, image_features):
        mapped_features = self.mlp(image_features)
        # 인스턴스 변수를 통해 gpt2_hidden_dim에 접근
        soft_prompt_embeddings = mapped_features.view(-1, self.num_soft_tokens, self.gpt2_hidden_dim)
        return soft_prompt_embeddings

CLIP_EMBEDDING_DIM = clip_model.config.projection_dim # CLIP ViT-L/14의 임베딩 차원 (768)
GPT2_HIDDEN_DIM = gpt2_model.config.hidden_size       # GPT-2 XL의 히든 스테이트 차원 (1600)

mapping_network = ImageToSoftPromptMapping(CLIP_EMBEDDING_DIM, GPT2_HIDDEN_DIM, NUM_SOFT_TOKENS).to(device)

# 매핑 네트워크의 학습 가능한 파라미터 수 확인 (총 3B 제한에 포함)
total_mapping_params = sum(p.numel() for p in mapping_network.parameters() if p.requires_grad)
print(f"매핑 네트워크 학습 가능한 파라미터: {total_mapping_params / 1e6:.2f}M")
print(f"전체 모델 총 파라미터: {(sum(p.numel() for p in clip_model.parameters()) + sum(p.numel() for p in gpt2_model.parameters()) + total_mapping_params) / 1e9:.3f}B")


In [ ]:
# --- 4. VMCBench 데이터셋 클래스 정의 ---
class VMCBenchDataset(Dataset):
    def __init__(self, annotation_file, question_file, image_dir, tokenizer, clip_processor, split_type):
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        self.clip_processor = clip_processor
        self.split_type = split_type # "train2014" or "val2014"

        with open(annotation_file, 'r', encoding='utf-8') as f:
            self.annotations = json.load(f)["annotations"]
        with open(question_file, 'r', encoding='utf-8') as f:
            self.questions_data = {q["question_id"]: q for q in json.load(f)["questions"]}

        self.annotations = [anno for anno in self.annotations if anno["question_id"] in self.questions_data]

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        anno = self.annotations[idx]
        question_id = anno["question_id"]
        image_id = anno["image_id"]
        
        question_info = self.questions_data[question_id]
        question_text = question_info["question"] # 질문 + A, B, C, D 보기가 이미 포함되어 있음

        # 이미지 로드
        image_filename = f"COCO_{self.split_type}_{str(image_id).zfill(12)}.jpg" # 동적 split_type 사용
        image_path = os.path.join(self.image_dir, image_filename)
        try:
            image = Image.open(image_path).convert("RGB")
        except FileNotFoundError:
            print(f"경고: 이미지 파일 {image_path}를 찾을 수 없습니다. None 반환.")
            return None # 이미지 파일을 찾을 수 없는 경우 None 반환
            

        # CLIP 이미지 전처리
        clip_image_inputs = self.clip_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

        # GPT-2 입력 프롬프트 구성: 질문 + 보기 + "Answer:"
        full_prompt = f"{question_text}\nAnswer:"
        gpt2_inputs = self.tokenizer(full_prompt, return_tensors="pt", padding=False, truncation=True, max_length=512)

        # GPT-2의 목표 정답 토큰 ID (예: 'A', 'B', 'C', 'D'의 토큰 ID)
        target_char_label = chr(ord('A') + anno["answer_label"])
        target_ids = self.tokenizer(target_char_label, return_tensors="pt")["input_ids"].squeeze(0)
        
        # GPT-2의 최종 입력 시퀀스와 레이블 구성
        # labels_for_collate는 GPT-2 텍스트 입력의 원래 토큰 ID와 정답 토큰 ID를 포함합니다.
        # collate_fn에서 이를 기반으로 최종 labels 텐서를 만듭니다.
        labels_for_collate_item = torch.cat([gpt2_inputs["input_ids"].squeeze(0), target_ids], dim=0)

        return {
            "clip_image_inputs": clip_image_inputs,
            "gpt2_input_ids_for_collate": gpt2_inputs["input_ids"].squeeze(0),
            "attention_mask_for_collate": gpt2_inputs["attention_mask"].squeeze(0),
            "labels_for_collate": labels_for_collate_item
        }

In [ ]:
# --- 5. Custom Collate Function ---
def collate_fn(batch):
    # __getitem__에서 None이 반환된 경우 해당 항목을 필터링합니다.
    batch = [item for item in batch if item is not None]
    if not batch: # 배치에 유효한 항목이 없으면 None 반환
        return None 
    
    clip_image_inputs = torch.stack([item['clip_image_inputs'] for item in batch])
    
    max_len_text_input = max(item['gpt2_input_ids_for_collate'].size(0) for item in batch)
    
    # 최종 시퀀스 길이: 소프트 토큰 개수 + GPT-2 텍스트 입력의 최대 길이
    # GPT-2 모델의 inputs_embeds와 labels의 두 번째 차원(sequence_length)과 정확히 일치해야 함.
    max_seq_len_for_inputs_and_labels = NUM_SOFT_TOKENS + max_len_text_input 

    padded_gpt2_input_ids = []
    padded_attention_mask = []
    padded_labels = []

    for item in batch:
        gpt2_input_ids_unpadded = item['gpt2_input_ids_for_collate'] # 현재 아이템의 패딩되지 않은 GPT-2 입력 ID
        attention_mask_unpadded = item['attention_mask_for_collate']
        target_token_id = item['labels_for_collate'][-1].item() # 현재 아이템의 정답 토큰 ID
        
        # 1. GPT-2 텍스트 입력 ID를 `max_len_text_input` 길이로 패딩
        pad_len_input = max_len_text_input - gpt2_input_ids_unpadded.size(0)
        padded_gpt2_input_ids_item = torch.cat([gpt2_input_ids_unpadded, torch.full((pad_len_input,), gpt2_tokenizer.pad_token_id, dtype=torch.long)], dim=0)
        padded_gpt2_input_ids.append(padded_gpt2_input_ids_item)
        
        # 2. GPT-2 텍스트 어텐션 마스크 패딩
        padded_attention_mask_item = torch.cat([attention_mask_unpadded, torch.zeros(pad_len_input, dtype=torch.long)], dim=0)
        padded_attention_mask.append(padded_attention_mask_item)
        
        # 3. 레이블 텐서 구성 (critical part)
        # `labels` 텐서는 `max_seq_len_for_inputs_and_labels` 길이여야 함.
        current_item_labels = torch.full((max_seq_len_for_inputs_and_labels,), -100, dtype=torch.long)
        
        # 정답 토큰을 배치할 위치를 결정.
        # 이는 GPT-2가 정답을 예측해야 하는 시점의 입력 토큰 위치입니다.
        # 입력 시퀀스는 `[소프트 토큰] + [질문+보기+Answer:]` 형태입니다.
        # `Answer:` 토큰은 `gpt2_input_ids_unpadded`의 마지막 유효 토큰입니다.
        # 이 `Answer:` 토큰에 해당하는 레이블이 `target_token_id`여야 합니다.
        # `Answer:` 토큰의 최종 시퀀스 내 인덱스는 `NUM_SOFT_TOKENS + (현재 아이템의 패딩되지 않은 텍스트 입력 길이) - 1` 입니다.
        
        answer_label_position = NUM_SOFT_TOKENS + gpt2_input_ids_unpadded.size(0) - 1

        # 해당 위치가 유효한지 확인 후 레이블 설정
        if answer_label_position >= 0 and \
           answer_label_position < max_seq_len_for_inputs_and_labels:
            current_item_labels[answer_label_position] = target_token_id
        else:
            # 이 경고가 자주 나타나면 `max_length` 또는 `NUM_SOFT_TOKENS` 설정 재검토 필요
            print(f"CRITICAL WARNING: 정답 레이블 위치({answer_label_position})가 시퀀스 길이({max_seq_len_for_inputs_and_labels})를 초과합니다. 이 항목의 레이블이 올바르게 설정되지 않을 수 있습니다.")
            print(f"GPT2 입력 길이: {gpt2_input_ids_unpadded.size(0)}, 소프트 토큰: {NUM_SOFT_TOKENS}, Max Text Input: {max_len_text_input}, 타겟 ID: {target_token_id}")

        padded_labels.append(current_item_labels)

    return {
        "clip_image_inputs": clip_image_inputs,
        "gpt2_input_ids": torch.stack(padded_gpt2_input_ids),
        "attention_mask": torch.stack(padded_attention_mask),
        "labels": torch.stack(padded_labels)
    }

In [ ]:
# --- 6. DataLoader 생성 ---
print("데이터로더를 생성 중입니다...")
train_dataset = VMCBenchDataset(
    annotation_file=TRAIN_ANNOTATION_FILE,
    question_file=TRAIN_QUESTION_FILE,
    image_dir=TRAIN_IMAGE_DIR,
    tokenizer=gpt2_tokenizer,
    clip_processor=clip_processor,
    split_type="train2014"
)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
print(f"훈련 데이터셋 크기: {len(train_dataset)}")
print(f"훈련 데이터로더 배치 수: {len(train_dataloader)}")

# 검증 데이터로더 (선택 사항이지만 강력 권장)
val_dataset = VMCBenchDataset(
    annotation_file=VAL_ANNOTATION_FILE,
    question_file=VAL_QUESTION_FILE,
    image_dir=VAL_IMAGE_DIR,
    tokenizer=gpt2_tokenizer,
    clip_processor=clip_processor,
    split_type="val2014"
)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"검증 데이터셋 크기: {len(val_dataset)}")
print(f"검증 데이터로더 배치 수: {len(val_dataloader)}")

In [28]:
# --- 7. 학습 루프 ---
print("매핑 네트워크 학습을 시작합니다...")
optimizer = torch.optim.AdamW(mapping_network.parameters(), lr=LEARNING_RATE) # 매핑 네트워크만 학습
scaler = torch.cuda.amp.GradScaler() # Mixed precision training (GPU 사용 시 추천)

best_val_loss = float('inf')
for epoch in range(NUM_EPOCHS):
    mapping_network.train()
    total_train_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} (Train)")

    for batch in progress_bar:
        # collate_fn에서 None이 반환될 수 있으므로, 여기서 None인 배치는 건너뜁니다.
        if batch is None:
            continue

        optimizer.zero_grad()

        clip_image_inputs = batch["clip_image_inputs"].to(device)
        gpt2_input_ids = batch["gpt2_input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        with torch.no_grad(): # CLIP은 고정되어 있으므로 no_grad 블록 사용
            image_features = clip_model.get_image_features(pixel_values=clip_image_inputs)
        
        # 매핑 네트워크를 통해 소프트 프롬프트 임베딩 생성
        soft_prompt_embeddings = mapping_network(image_features)

        # GPT-2 텍스트 입력의 임베딩 가져오기
        input_embeddings = gpt2_model.get_input_embeddings()(gpt2_input_ids)
        
        # 소프트 프롬프트 임베딩과 텍스트 입력 임베딩을 결합 (소프트 프롬프트가 앞에 위치)
        extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
        
        # 어텐션 마스크 확장
        soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
        extended_attention_mask = torch.cat((soft_prompt_attention_mask, attention_mask), dim=1)
        
        # Mixed precision training (optional)
        with torch.cuda.amp.autocast():
            outputs = gpt2_model(
                inputs_embeds=extended_input_embeddings,
                attention_mask=extended_attention_mask,
                labels=labels # GPT-2 will compute loss with these labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_train_loss += loss.item()
        progress_bar.set_postfix(loss=total_train_loss / (progress_bar.n + 1))

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.4f}")

    # --- 검증 루프 (Validation Loop) ---
    mapping_network.eval() # 평가 모드
    total_val_loss = 0
    val_progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} (Validation)")
    with torch.no_grad():
        for batch in val_progress_bar:
            if batch is None:
                continue

            clip_image_inputs = batch["clip_image_inputs"].to(device)
            gpt2_input_ids = batch["gpt2_input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            image_features = clip_model.get_image_features(pixel_values=clip_image_inputs)
            soft_prompt_embeddings = mapping_network(image_features)
            input_embeddings = gpt2_model.get_input_embeddings()(gpt2_input_ids)
            extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
            
            soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
            extended_attention_mask = torch.cat((soft_prompt_attention_mask, attention_mask), dim=1)

            outputs = gpt2_model(
                inputs_embeds=extended_input_embeddings,
                attention_mask=extended_attention_mask,
                labels=labels
            )
            loss = outputs.loss
            total_val_loss += loss.item()
            val_progress_bar.set_postfix(loss=total_val_loss / (val_progress_bar.n + 1))

    avg_val_loss = total_val_loss / len(val_dataloader)
    print(f"Epoch {epoch+1} Validation Loss: {avg_val_loss:.4f}")

    # 모델 저장 (검증 손실이 개선될 때만)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        mapping_network_save_path = "best_mapping_network_weights.pth"
        torch.save(mapping_network.state_dict(), mapping_network_save_path)
        print(f"검증 손실 개선! 모델 저장됨: {mapping_network_save_path}")

print("매핑 네트워크 학습 완료.")



설정을 시작합니다...
사용할 장치: cuda
모델 및 토크나이저를 로드 중입니다...
CLIP 모델 로드 완료. 파라미터 수: 0.428B
GPT-2 모델 로드 완료. 파라미터 수: 1.558B


/tmp/ipykernel_3591593/1825620199.py:249: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # Mixed precision training (GPU 사용 시 추천)


매핑 네트워크 학습 가능한 파라미터: 524.34M
전체 모델 총 파라미터: 2.510B
데이터로더를 생성 중입니다...
훈련 데이터셋 크기: 888
훈련 데이터로더 배치 수: 444
검증 데이터셋 크기: 100
검증 데이터로더 배치 수: 50
매핑 네트워크 학습을 시작합니다...


Epoch 1/5 (Train):   0%|          | 0/444 [00:00<?, ?it/s]/tmp/ipykernel_3591593/1825620199.py:286: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/5 (Train): 100%|██████████| 444/444 [01:28<00:00,  5.00it/s, loss=2.01]


Epoch 1 Train Loss: 2.0060


...

Epoch 4 Validation Loss: 2.8031


Epoch 5/5 (Train): 100%|██████████| 444/444 [01:15<00:00,  5.89it/s, loss=0.327]


Epoch 5 Train Loss: 0.3274


Epoch 5/5 (Validation): 100%|██████████| 50/50 [00:06<00:00,  7.95it/s, loss=2.74]

Epoch 5 Validation Loss: 2.6899
매핑 네트워크 학습 완료.


In [ ]:
# --- 8. 추론 함수 (학습된 모델 사용) ---
print("\n--- 추론 예시 ---")

# 모델 로드 (학습된 매핑 네트워크 가중치)
loaded_mapping_network = ImageToSoftPromptMapping(CLIP_EMBEDDING_DIM, GPT2_HIDDEN_DIM, NUM_SOFT_TOKENS).to(device)
loaded_mapping_network.load_state_dict(torch.load("best_mapping_network_weights.pth")) # 가장 좋은 가중치 로드
loaded_mapping_network.eval() # 추론 모드

def predict_answer(image_path, question, choices):
    # 이미지 로드 및 CLIP 임베딩 추출
    image = Image.open(image_path).convert("RGB")
    clip_inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_features = clip_model.get_image_features(pixel_values=clip_inputs["pixel_values"])
    
    # 매핑 네트워크를 통해 소프트 프롬프트 임베딩 생성
    with torch.no_grad():
        soft_prompt_embeddings = loaded_mapping_network(image_features)

    # GPT-2 프롬프트 구성 (질문 + 보기)
    choices_text = [
        f"A. {choices[0]}",
        f"B. {choices[1]}",
        f"C. {choices[2]}",
        f"D. {choices[3]}"
    ]
    
    joined_choices_text = '\n'.join(choices_text)
    full_question_text = f"{question}\n{joined_choices_text}\nAnswer:"

    gpt2_text_inputs = gpt2_tokenizer(full_question_text, return_tensors="pt").to(device)

    # GPT-2 텍스트 입력의 임베딩 가져오기
    input_embeddings = gpt2_model.get_input_embeddings()(gpt2_text_inputs["input_ids"])

    # 소프트 프롬프트와 텍스트 임베딩 결합
    extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
    
    # 어텐션 마스크 확장
    soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
    extended_attention_mask = torch.cat((soft_prompt_attention_mask, gpt2_text_inputs["attention_mask"]), dim=1)

    # GPT-2 답변 생성 (다음 토큰을 예측하도록)
    with torch.no_grad():
        output_ids = gpt2_model.generate(
            inputs_embeds=extended_input_embeddings,
            attention_mask=extended_attention_mask,
            max_new_tokens=1, # 오직 하나의 정답 문자만 생성
            num_beams=1,      # 빔 서치 사용 안함
            do_sample=False,  # 샘플링 사용 안함 (가장 높은 확률 토큰 선택)
            pad_token_id=gpt2_tokenizer.pad_token_id,
            eos_token_id=gpt2_tokenizer.eos_token_id # EOS 토큰 만나면 생성 중단
        )
    
    # 생성된 토큰에서 원래 프롬프트 부분 제거 후 디코딩
    generated_token_id = output_ids[0, extended_input_embeddings.shape[1]]
    predicted_char = gpt2_tokenizer.decode(generated_token_id, skip_special_tokens=True).strip()

    # 결과가 'A', 'B', 'C', 'D' 중 하나가 아닐 경우 처리 (Fallback)
    if predicted_char not in ['A', 'B', 'C', 'D']:
        print(f"경고: 예상치 못한 답변 생성: '{predicted_char}'. 랜덤 선택으로 대체합니다.")
        return random.choice(['A', 'B', 'C', 'D']) # Fallback to random choice if invalid char is generated

    return predicted_char

# --- 9. 대회 테스트 데이터셋으로 추론 ---
# 대회에서 제공된 test dataset 경로를 사용합니다.
TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test.csv"
TEST_IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test_input_images"

df_test_input = pd.read_csv(TEST_CSV_PATH)
df_submission = pd.DataFrame(columns=["ID", "answer"])

print("\n대회 테스트 데이터셋으로 추론 시작...")
for index, row in tqdm(df_test_input.iterrows(), total=len(df_test_input), desc="Generating predictions"):
    img_id = row['ID']
    img_path = os.path.join(TEST_IMAGE_DIR, os.path.basename(row['img_path']))
    question = row['Question']
    choices = [row['A'], row['B'], row['C'], row['D']]

    predicted_answer = predict_answer(img_path, question, choices)
    
    df_submission = pd.concat([df_submission, pd.DataFrame([{"ID": img_id, "answer": predicted_answer}])], ignore_index=True)

# 최종 제출 파일 저장
SUBMISSION_FILE_PATH = "gpt_clip.csv"
df_submission.to_csv(SUBMISSION_FILE_PATH, index=False)
print(f"\n최종 제출 파일 저장 완료: {SUBMISSION_FILE_PATH}")


--- 추론 예시 ---

대회 테스트 데이터셋으로 추론 시작...


Generating predictions:   0%|          | 0/852 [00:00<?, ?it/s]


IndexError: index 76 is out of bounds for dimension 1 with size 1

: 